In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
indicator_name = 'Accessibility_4'
# estimate = 'ACS1'
estimate = 'ACS5'


In [ ]:
df_acs_p = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_P.csv')
                                    , dtype = {'State FIPS': object, 'PUMA': object, 'SERIALNO': object})
df_acs_p = df_acs_p[df_acs_p['SPORDER'] == 1]
df_acs_p = df_acs_p[df_acs_p['JWTRNS'] != 'N/A, not a civillian in the labor force']

df_acs_h = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_H.csv')
                                    , dtype = {'State FIPS': object, 'PUMA': object, 'SERIALNO': object})
df_acs_h = df_acs_h[(df_acs_h['NP'] > 0) & (df_acs_h['WGTP'] > 0)]


df_cpi = pd.read_excel(os.path.join(path_config0, 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')
df_cpi = df_cpi[['Year', 'IAF_2023']].rename(columns = {'Year':'year'})


In [ ]:
df_acs_p.head()

In [ ]:
df_acs_h.head()

In [ ]:
df_cpi.head()

In [ ]:
df_acs_h = df_acs_h.merge(df_acs_p.drop(['SPORDER', 'PWGTP'], axis = 1), on = ['State FIPS', 'PUMA', 'PUMA NAME', 'SERIALNO', 'year'], how = 'left')
df_acs_h = df_acs_h.dropna(subset = ['JWTRNS'])

df_acs_h = df_acs_h.merge(df_cpi, on = 'year', how = 'left')
df_acs_h['HINCP'] = df_acs_h['HINCP']*df_acs_h['ADJINC']*df_acs_h['IAF_2023']
df_acs_h = df_acs_h.drop(['IAF_2023', 'ADJINC'], axis = 1)

df_acs_h.head()

In [ ]:
df_acs = df_acs_h.copy()

df_fips1 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'CountyFIPS' 
                                , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips2 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'PUMAcodes' 
                                , dtype = {'STATEFP': object, 'COUNTYFP': object})

df_fips1 = df_fips1[df_fips1['State FIPS'].isin(['06'])]
df_fips2 = df_fips2[df_fips2['STATEFP'   ].isin(['06'])]

df_fips2['PUMA5CE'] = df_fips2['PUMA5CE'].astype(str).apply('{:0>5}'.format)
df_fips2 = df_fips2[['STATEFP', 'COUNTYFP', 'PUMA5CE', 'Years']].drop_duplicates()
df_fips2 = df_fips2.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS', 'PUMA5CE':'PUMA'})

df_acs['PUMA'] = df_acs['PUMA'].astype(str).apply('{:0>5}'.format)

df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MPO']]
# df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MSA']]

df_acs2 = df_acs[df_acs['year'].isin(sequence(2022, 2031, 1))].merge(df_fips2[df_fips2['Years'] == '2022-2031'], on = ['State FIPS', 'PUMA'])
df_acs1 = df_acs[df_acs['year'].isin(sequence(2012, 2021, 1))].merge(df_fips2[df_fips2['Years'] == '2012-2021'], on = ['State FIPS', 'PUMA'])
df_acs = pd.concat([df_acs1, df_acs2])

df_acs = df_acs.merge(df_fips1, on = ['State FIPS', 'County FIPS'])
df_acs = df_acs.drop(['SERIALNO', 'Years'], axis = 1)

df_acs.head(3)

In [ ]:
df_income_brackets = pd.read_excel(os.path.join(path_config0, 'CA State Income Brackets by Household Size.xlsx'), sheet_name = 'Table')
df_income_brackets['County'].fillna(method='ffill', inplace = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace(' County.*'         , '' , regex = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace('\n'                , ' ', regex = True)
df_income_brackets['AMI'   ] = df_income_brackets['County'].str.extract('\$?([0-9,]+)[.%]?')
df_income_brackets['AMI'   ] = df_income_brackets['AMI'   ].str.replace(','                 , '' , regex = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace(' \$?([0-9,]+)[.%]?', '' , regex = True)
df_income_brackets = pd.melt(df_income_brackets
                              , id_vars = ['County', 'Income Bracket', 'AMI']
                              , var_name = 'NP'
                              , value_name = 'Income Threshold'
                            )
df_income_brackets = df_income_brackets[df_income_brackets['Income Bracket'].isin(['Low Income', 'Moderate Income'])]
df_income_brackets = df_income_brackets.pivot_table(index = ['County', 'NP']
                                       , columns = 'Income Bracket'
                                       , values = 'Income Threshold').reset_index().rename(columns = {'County':'County Name'})
df_income_brackets.head()

In [ ]:
df_acs = df_acs.merge(df_income_brackets, on = ['County Name', 'NP'], how = 'left')
df_acs.head()

In [ ]:
df_acs.loc[ df_acs['HINCP'] <= df_acs['Low Income']                                                  , 'Income Bracket'] = 'Low Income'
df_acs.loc[(df_acs['HINCP']  > df_acs['Low Income']) & (df_acs['HINCP'] <= df_acs['Moderate Income']), 'Income Bracket'] = 'Moderate Income'
df_acs.loc[ df_acs['HINCP']  > df_acs['Moderate Income']                                             , 'Income Bracket'] = 'High Income'
df_acs.loc[ df_acs['NP'] == 0                                                                        , 'Income Bracket'] = 'No data available'

df_acs.loc[(df_acs['JWMNP'] ==  0)                          , 'Travel Time'] = 'No commute (worked from home)'
df_acs.loc[(df_acs['JWMNP']  >  0) & (df_acs['JWMNP'] <= 15), 'Travel Time'] = '0 to 15 minutes'
df_acs.loc[(df_acs['JWMNP']  > 15) & (df_acs['JWMNP'] <= 30), 'Travel Time'] = '15 to 30 minutes'
df_acs.loc[(df_acs['JWMNP']  > 30)                          , 'Travel Time'] = 'More than 30 minutes'
df_acs.head()

In [ ]:
df_acs1 = df_acs.groupby(['State FIPS', 'County Name', 'year', 'RAC1P', 'Income Bracket', 'Travel Time'], as_index = False)['WGTP'].sum()
df_acs1_all = df_acs1.groupby(['State FIPS', 'County Name', 'year', 'Income Bracket', 'Travel Time'], as_index = False)['WGTP'].agg(sum)
df_acs1_all['RAC1P'     ] = 'All'
df_acs1_all['Percentage'] = np.nan
df_acs1 = pd.concat([df_acs1, df_acs1_all])

df_mpo1 = df_acs.groupby(['State FIPS', 'MPO', 'year', 'RAC1P', 'Income Bracket', 'Travel Time'], as_index = False)['WGTP'].sum()
df_mpo1_all = df_mpo1.groupby(['State FIPS', 'MPO', 'year', 'Income Bracket', 'Travel Time'], as_index = False)['WGTP'].agg(sum)
df_mpo1_all['RAC1P'     ] = 'All'
df_mpo1_all['Percentage'] = np.nan
df_mpo1 = pd.concat([df_mpo1, df_mpo1_all])


df_acs1['Percentage'] = 100*df_acs1['WGTP'] / df_acs1.groupby(['County Name', 'year',  'RAC1P', 'Income Bracket'])['WGTP'].transform('sum')
df_mpo1['Percentage'] = 100*df_mpo1['WGTP'] / df_mpo1.groupby(['MPO', 'year', 'RAC1P', 'Income Bracket'])['WGTP'].transform('sum')


# Create "Categorical" race/ethnicity field for sorting
# Sort by geography, variable mapping, and race/ethnicity
# sort and then remove categorical field
df_acs1['Income_sort'] = pd.Categorical(df_acs1['Income Bracket'], ['No data available'
                                                                     , 'Low Income'
                                                                     , 'Moderate Income'
                                                                     , 'High Income'
                                                                    ])

df_acs1['RAC1P_sort'] = pd.Categorical(df_acs1['RAC1P'], ['All'
                                                            , 'American Indian or Alaska Native (NH)'
                                                            , 'Asian (NH)'
                                                            , 'Black or African American (NH)'
                                                            , 'Hispanic or Latino'
                                                            , 'Native Hawaiian or other Pacific Islander (NH)'
                                                            , 'White (NH)'
                                                            , 'Some other race (NH)'
                                                            , 'Two or more races (NH)'
                                                         ])

df_acs1['Travel_sort'] = pd.Categorical(df_acs1['Travel Time'], ['No commute  (worked from home)'
                                                                    , '0 to 15 minutes'
                                                                    , '15 to 30 minutes'
                                                                    , 'More than 30 minutes'
                                                                    ])



df_mpo1['Income_sort'] = pd.Categorical(df_mpo1['Income Bracket'], ['No data available'
                                                                     , 'Low Income'
                                                                     , 'Moderate Income'
                                                                     , 'High Income'
                                                                    ])

df_mpo1['RAC1P_sort'] = pd.Categorical(df_mpo1['RAC1P'], ['All'
                                                            , 'American Indian or Alaska Native (NH)'
                                                            , 'Asian (NH)'
                                                            , 'Black or African American (NH)'
                                                            , 'Hispanic or Latino'
                                                            , 'Native Hawaiian or other Pacific Islander (NH)'
                                                            , 'White (NH)'
                                                            , 'Some other race (NH)'
                                                            , 'Two or more races (NH)'
                                                         ])

df_mpo1['Travel_sort'] = pd.Categorical(df_mpo1['Travel Time'], ['No commute (worked from home)'
                                                                    , '0 to 15 minutes'
                                                                    , '15 to 30 minutes'
                                                                    , 'More than 30 minutes'
                                                                    ])



df_acs1 = df_acs1.sort_values(by = ['County Name', 'year', 'Income_sort', 'RAC1P_sort', 'Travel_sort'], ascending = [True, False, True, True, True])
df_mpo1 = df_mpo1.sort_values(by = ['MPO'        , 'year', 'Income_sort', 'RAC1P_sort', 'Travel_sort'], ascending = [True, False, True, True, True])
df_acs1 = df_acs1.drop(['RAC1P_sort', 'Income_sort', 'Travel_sort'], axis = 1)
df_mpo1 = df_mpo1.drop(['RAC1P_sort', 'Income_sort', 'Travel_sort'], axis = 1)
df_mpo1

In [ ]:
geography = 'PUMA'

# Set output name for .xlsx files
name_output_xlsx     = [indicator_name, ' PUMS County ', estimate, '.xlsx']
name_output_MPO_xlsx = [indicator_name, ' PUMS MPO '   , estimate, '.xlsx']
name_output_xlsx     = "".join(name_output_xlsx    )
name_output_MPO_xlsx = "".join(name_output_MPO_xlsx)

# Set output name for .csv files
name_output_PUMA_csv = [indicator_name, '_PUMA_', estimate, '.csv']
name_output_PUMA_csv = "".join(name_output_PUMA_csv)

In [ ]:
report_theme = 'Next Gen of Mobility Solutions'
sp_folder_out = 'Travel Modes\\Accessibility_4 Commute Times'


# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out)
path_out_csv  = os.path.join(path_agol, indicator_name)

with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
# with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_acs1.to_excel(writer, index = False, sheet_name = 'Counties')

with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_MPO_xlsx), engine='xlsxwriter') as writer:
# with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_MPO_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_mpo1.to_excel(writer, index = False, sheet_name = 'MPO')
# df_acs1_csv.to_csv(os.path.join(path_out_csv, name_output_county_csv), index = False)
# df_mpo1_csv.to_csv(os.path.join(path_out_csv, name_output_MPO_csv   ), index = False)



print('')
print("Successfully exported")

In [ ]:
print('Columns: ' + str(list(df_mpo1.columns)))

In [ ]:
df_plot = df_mpo1[
                    (df_mpo1['Income Bracket'].isin(['Low Income', 'Moderate Income', 'High Income']))
                    & (df_mpo1['RAC1P'] == 'All')
                     
                     ]
# df_plot = df_plot[df_plot['County Name'] == 'Placer']

# Plotting setup
by_race = False
race_ethnicity = 'RAC1P'
by_vars = False
variable = 'Income Bracket'
x = 'year'
y = 'Percentage'
line_dash = 'Travel Time'
color = 'Income Bracket'

markers = True
# plot_title = 'Commute Modes by Income Level'
plot_title = 'Number of Households by Income Level'
# plot_name = 'Median Household Income by Peer MSA'
plot_name = 'Number of Households by Income Level'
export = False

In [ ]:
def plot_lines(
    df=df_plot
     , by_vars=by_vars, by_race=by_race, race_ethnicity=race_ethnicity, variable=variable
     , x=x, y=y
     , color=color, line_dash=line_dash, markers=markers
     , plot_title=plot_title, plot_name=plot_name
     , export=export
):

    if by_vars == True:
        vars = unique(df[variable].values)
        for var in vars:
            df2 = df[df[variable] == var]
            fig = px.line(df2, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
            fig.update_layout(title = plot_title + ' - ' + str(var))
            if export == True:
                fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', var + '_', 'line.html'])))
                
            fig.update_layout(autosize=False, width=1050, height=450)
            
    else:
        fig = px.line(df, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
        fig.update_layout(title = plot_title)
        if export == True:
            fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', 'line.html'])))

        fig.update_layout(autosize=False, width=1050, height=450)

    return fig.show()
        
plot_lines()

In [ ]:
df_rep = df_acs[['State FIPS', 'MPO', 'County Name', 'year', 'RAC1P', 'Income Bracket', 'WGTP', 'JWMNP']]
df_rep = df_rep.groupby(['State FIPS', 'MPO', 'year','Income Bracket', 'JWMNP'], as_index = False)['WGTP'].agg(sum)
df_rep.head()

In [ ]:
list_keys = []
for list_ in df_rep.values:
    list_keys.append(tuple(list_[:-1]))

list_values = []
for list_ in df_rep.values:
    list_values.append(list_[-1])

dict_replicates = dict(zip(list_keys, list_values))

list_df = []
for tuple_ in tqdm(list(dict_replicates.keys())):
    
    row  = list(tuple_)
    wgtp = dict_replicates[tuple_]

    list_df.append(pd.concat([pd.DataFrame(row).T] * wgtp))

df_rep = pd.concat(list_df)
df_rep.columns = ['State FIPS', 'MPO',  'year', 'Income Bracket', 'JWMNP']
df_rep.head()

In [ ]:
df_plot = df_rep[df_rep['year'].isin([2019, 2020, 2021])] 
df_plot = df_plot[df_plot['Income Bracket'] == 'High Income']
fig = px.histogram(df_plot, x="JWMNP", facet_col='year', nbins=40)
fig.show()

In [ ]:
# df_plot = df_rep.copy()
# fig = px.box(df_plot[df_plot['JWMNP'] < 100], x='year', y='JWMNP')
# fig.show()